# Survival Fairness Analysis

Bootstrap-based fairness analysis for survival models across sites and demographic groups.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import joblib
from fairness_utils import (
    load_predictions_and_data,
    load_model,
    compare_cindex_by_site,
    compare_cindex_by_race,
    compare_cindex_by_groups,
    compute_lipi
)

## Configuration

In [ ]:
# Analysis settings
outcome = "OS_24"
analysis = "C23"
modality = "RWD"
base_path = "MLEF"

# Sites for fairness analysis
sites = ['INT', 'MH', 'GHD', 'VHIO', 'SZMC']

# Bootstrap settings
n_boot = 2000
seed = 42

## Load Data and Model

In [ ]:
# Load predictions and test data
predictions_df, test_df = load_predictions_and_data(
    outcome=outcome,
    analysis=analysis,
    modality=modality,
    base_path=base_path
)

# Load trained model
model = load_model(
    outcome=outcome,
    analysis=analysis,
    modality=modality,
    base_path=base_path
)

## Fairness Analysis by Site

In [ ]:
# Compare C-index across sites
summary_site, pairwise_site = compare_cindex_by_site(
    test_df=test_df,
    model=model,
    sites=sites,
    n_boot=n_boot
)

print("\n=== C-index by Site ===")
print(summary_site.to_string(index=False))

print("\n=== Pairwise Comparisons (Holm-adjusted) ===")
print(pairwise_site[['g1', 'g2', 'diff', 'diff_lo', 'diff_hi', 'p', 'p_holm']].to_string(index=False))

## Fairness Analysis by Race

In [ ]:
# Load race metadata
race_metadata = pd.read_excel('../data/child23_imp_enc.xlsx')

# Define race columns
race_cols = {
    'C112_RACE_WHITE': 'WHITE',
    'C112_RACE_BLACK OR AFRICAN AMERICAN': 'BLACK'
}

# Compare C-index across races
summary_race, pairwise_race = compare_cindex_by_race(
    test_df=test_df,
    model=model,
    race_metadata=race_metadata,
    race_cols=race_cols,
    n_boot=n_boot
)

print("\n=== C-index by Race ===")
print(summary_race.to_string(index=False))

print("\n=== Pairwise Comparisons (Holm-adjusted) ===")
print(pairwise_race[['g1', 'g2', 'diff', 'diff_lo', 'diff_hi', 'p', 'p_holm']].to_string(index=False))

## Fairness Analysis by Gender

In [ ]:
# Assuming test_df has a 'GENDER_M' column
if 'GENDER_M' in test_df.columns:
    test_df_gender = test_df.copy()
    test_df_gender['gender'] = test_df_gender['GENDER_M'].map({1: 'Male', 0: 'Female'})
    
    summary_gender, pairwise_gender = compare_cindex_by_groups(
        test_df=test_df_gender,
        model=model,
        group_col='gender',
        n_boot=n_boot
    )
    
    print("\n=== C-index by Gender ===")
    print(summary_gender.to_string(index=False))
    
    print("\n=== Pairwise Comparisons ===")
    print(pairwise_gender[['g1', 'g2', 'diff', 'diff_lo', 'diff_hi', 'p', 'p_holm']].to_string(index=False))
else:
    print("GENDER_M column not found in test data")

## LIPI Analysis

In [ ]:
# Compute LIPI if lab values available
required_cols = ['LAB22_LDH_BASAL', 'LAB15_LEUKOCYTES_BASAL', 'LAB16_NEUTROPHYL_BASAL']

if all(col in test_df.columns for col in required_cols):
    test_df['LIPI'] = compute_lipi(test_df)
    
    summary_lipi, pairwise_lipi = compare_cindex_by_groups(
        test_df=test_df,
        model=model,
        group_col='LIPI',
        n_boot=n_boot
    )
    
    print("\n=== C-index by LIPI ===")
    print(summary_lipi.to_string(index=False))
    
    print("\n=== Pairwise Comparisons ===")
    print(pairwise_lipi[['g1', 'g2', 'diff', 'diff_lo', 'diff_hi', 'p', 'p_holm']].to_string(index=False))
else:
    print(f"Required lab columns not found: {required_cols}")

## Export Risk Scores with Metadata

In [ ]:
# Export predictions with risk scores and metadata
export_df = test_df.reset_index()[['Subject', 'y_pred', 'EVENT', 'TIME']].copy()

# Add LIPI if computed
if 'LIPI' in test_df.columns:
    export_df['LIPI'] = test_df['LIPI'].values

# Add site
export_df['site'] = export_df['Subject'].apply(
    lambda x: next((s for s in sites if str(x).startswith(s)), None)
)

# Save
output_path = f'risk_scores_{outcome}_{analysis}_{modality}.xlsx'
export_df.to_excel(output_path, index=False)
print(f"\nExported risk scores to: {output_path}")

## Summary Statistics

In [ ]:
# Overall summary
print("\n=== Overall Summary ===")
print(f"Total samples: {len(test_df)}")
print(f"Events: {test_df['EVENT'].sum()} ({100*test_df['EVENT'].mean():.1f}%)")
print(f"Median survival time: {test_df['TIME'].median():.1f}")

# Sample distribution by site
if 'site' in export_df.columns:
    print("\n=== Samples by Site ===")
    print(export_df['site'].value_counts().sort_index())